# 3.32 — Passive-Aggressive Algorithms

Passive-aggressive (PA) algorithms are online margin learners: each new labeled example either already satisfies a safety margin, or the model moves just far enough to fix that violation. In this lesson, we build the update from scratch with NumPy, inspect the hinge-loss arithmetic, and connect the raw margin rule to cost, validation gaps, and stabilized decisions.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build passive-aggressive learning one idea at a time. Run each cell in order and read the printed intermediate values — every margin, loss, and update size is shown. This walkthrough is self-contained, uses only NumPy and Matplotlib, and uses a `_w` suffix so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, dot products, norms, and deterministic toy data.
import matplotlib.pyplot as plt  # compact visualizations for margins and updates.
np.random.seed(0)  # reproducibility for the tiny online streams.

### 1. Online linear classification and the signed margin

Passive-aggressive learning sees one example at a time. The model is a weight vector `w`; the score is `w · x`; and a label `y` is either `+1` or `-1`. The important quantity is the **signed margin** `y * (w · x)`: it is positive when the prediction has the right sign, and it is at least `1` when the example is not merely correct but safely separated.

In [ ]:
X_w = np.array([[2.0, 1.0], [1.0, 2.0], [-2.0, -1.0], [-1.0, -2.0]])  # four points in two classes.
y_w = np.array([1.0, 1.0, -1.0, -1.0])  # labels are signs, not 0/1.
w_w = np.array([0.2, -0.1])  # a weak current classifier.
scores_w = X_w @ w_w  # raw decision scores.
margins_w = y_w * scores_w  # signed margins: correct side and confidence in one number.
print("scores:", np.round(scores_w, 3))
print("signed margins:", np.round(margins_w, 3))
assert np.allclose(np.round(margins_w, 3), [0.3, 0.0, 0.3, 0.0])

▶ What you'll see: the signs are mostly correct, but the margins are far below 1, so PA still considers them unsafe.

In [ ]:
plt.figure(figsize=(4.5, 3.5))
plt.scatter(X_w[y_w == 1, 0], X_w[y_w == 1, 1], color="teal", label="y=+1")
plt.scatter(X_w[y_w == -1, 0], X_w[y_w == -1, 1], color="crimson", label="y=-1")
grid_w = np.linspace(-2.5, 2.5, 100)
if abs(w_w[1]) > 1e-12:
    plt.plot(grid_w, -(w_w[0] / w_w[1]) * grid_w, color="black", label="w·x=0")
plt.axhline(0, color="gray", linewidth=0.5); plt.axvline(0, color="gray", linewidth=0.5)
plt.title("1: current linear separator")
plt.legend(); plt.show()

▶ What you'll see: a tentative decision boundary; correct signs alone do not guarantee a safe margin.

*Why it's done this way:* the signed margin combines correctness and confidence. A point with margin `0.1` is technically on the right side but fragile; a tiny perturbation could flip it. PA therefore uses margin `1` as the target contract: no update if `y wᵀx ≥ 1`, and a correction otherwise.

### 2. Hinge loss: the size of the current violation

The PA loss for one example is the hinge loss `max(0, 1 - y w·x)`. It is zero once the margin is at least 1, and grows linearly when the example is wrong or too close to the boundary. This loss is the amount of margin debt the update tries to repay.

In [ ]:
x_w = np.array([2.0, 1.0])  # one positive example.
y_one_w = 1.0
score_one_w = float(w_w @ x_w)
margin_one_w = y_one_w * score_one_w
loss_one_w = max(0.0, 1.0 - margin_one_w)
print("score:", round(score_one_w, 3), "margin:", round(margin_one_w, 3), "hinge loss:", round(loss_one_w, 3))
assert round(loss_one_w, 3) == 0.700

▶ What you'll see: a correct but under-confident positive example still has loss 0.7.

In [ ]:
margin_grid_w = np.linspace(-1.5, 2.5, 100)
hinge_grid_w = np.maximum(0.0, 1.0 - margin_grid_w)
plt.figure(figsize=(4.5, 3))
plt.plot(margin_grid_w, hinge_grid_w, color="purple")
plt.axvline(1.0, color="black", linestyle="--", label="safe margin")
plt.scatter([margin_one_w], [loss_one_w], color="red", zorder=3)
plt.title("2: hinge loss is margin debt")
plt.xlabel("signed margin y w·x"); plt.ylabel("max(0, 1 - margin)")
plt.legend(); plt.show()

▶ What you'll see: the curve is flat at zero after margin 1 and linear before it.

*Why it's done this way:* PA does not optimize a smooth probability; it enforces a geometric promise. The hinge loss measures exactly how short the current margin is. That makes the later update interpretable: choose the smallest movement that pays off this debt.

### 3. The passive-aggressive step size τ

The classic PA update is `w_new = w + τ y x`, where `τ = loss / ||x||²`. The numerator says how much margin debt remains; the denominator says how much a unit step along `y x` changes the margin. Dividing by `||x||²` makes the step just large enough to reach margin 1 for this example.

In [ ]:
norm_sq_w = float(x_w @ x_w)
tau_w = loss_one_w / norm_sq_w
w_new_w = w_w + tau_w * y_one_w * x_w
new_margin_w = float(y_one_w * (w_new_w @ x_w))
print("||x||^2:", round(norm_sq_w, 3), "tau:", round(tau_w, 3))
print("new w:", np.round(w_new_w, 3), "new margin:", round(new_margin_w, 3))
assert round(tau_w, 3) == 0.140
assert round(new_margin_w, 3) == 1.000

▶ What you'll see: the update moves the margin from 0.3 exactly to 1.0.

In [ ]:
plt.figure(figsize=(4.5, 3.5))
plt.quiver([0], [0], [w_w[0]], [w_w[1]], angles="xy", scale_units="xy", scale=1, color="gray", label="old w")
plt.quiver([0], [0], [w_new_w[0]], [w_new_w[1]], angles="xy", scale_units="xy", scale=1, color="teal", label="new w")
plt.quiver([w_w[0]], [w_w[1]], [w_new_w[0]-w_w[0]], [w_new_w[1]-w_w[1]], angles="xy", scale_units="xy", scale=1, color="red", label="τ y x")
plt.xlim(-0.1, 0.6); plt.ylim(-0.2, 0.2)
plt.title("3: smallest corrective movement")
plt.legend(); plt.show()

▶ What you'll see: `w` moves in the direction of `y x`, and only as far as needed for this example.

*Why it's done this way:* a step `α y x` changes the margin by `α ||x||²`. To erase loss `1 - y wᵀx`, solve `margin + α||x||² = 1`, giving `α = loss / ||x||²`. That is why PA is aggressive enough to fix the current point but passive about every unnecessary extra movement.

### 4. Capped variants: PA-I and PA-II as stability knobs

Real data can be noisy or non-separable. If PA always forces every example to margin 1, a mislabeled or extreme point can yank the weights too far. PA-I caps the step with `C`: `τ = min(C, loss / ||x||²)`. PA-II softens with `τ = loss / (||x||² + 1/(2C))`. Both are regularization-style stability knobs.

In [ ]:
x_noisy_w = np.array([0.2, 0.1])  # small norm means an uncapped step would be huge.
y_noisy_w = 1.0
w_bad_w = np.array([-1.0, -1.0])
loss_noisy_w = max(0.0, 1.0 - y_noisy_w * float(w_bad_w @ x_noisy_w))
norm_noisy_w = float(x_noisy_w @ x_noisy_w)
tau_raw_w = loss_noisy_w / norm_noisy_w
C_w = 2.0
tau_pai_w = min(C_w, tau_raw_w)
tau_paii_w = loss_noisy_w / (norm_noisy_w + 1.0 / (2.0 * C_w))
print("loss:", round(loss_noisy_w, 3), "||x||^2:", round(norm_noisy_w, 3))
print("raw tau:", round(tau_raw_w, 3), "PA-I:", round(tau_pai_w, 3), "PA-II:", round(tau_paii_w, 3))
assert round(tau_raw_w, 3) == 26.000
assert round(tau_pai_w, 3) == 2.000
assert round(tau_paii_w, 3) == 4.333

▶ What you'll see: the uncapped step would be 26, while the stabilized variants are much smaller.

In [ ]:
labels_w = ["raw PA", "PA-I cap", "PA-II soft"]
taus_w = [tau_raw_w, tau_pai_w, tau_paii_w]
plt.figure(figsize=(4.7, 3))
plt.bar(labels_w, taus_w, color=["crimson", "teal", "orange"])
plt.title("4: stability knobs shrink extreme updates")
plt.ylabel("step size τ"); plt.xticks(rotation=15)
plt.show()

▶ What you'll see: capping or softening dramatically reduces the response to a tiny-norm noisy point.

*Why it's done this way:* `C` plays the same role as a cost or regularization guardrail in model selection. The raw update optimizes the current example completely; the capped versions trade a remaining violation for smaller movement, which is often better for future data.

### 5. Training over a stream and watching empirical risk

A PA learner trains by scanning a stream, computing each example's hinge loss, and updating only on violations. The empirical risk is the average loss over a dataset. The lesson's verified toy arithmetic uses losses `0.235`, `0.070`, and `0.505`, whose average is `0.270`; below we also watch a real PA loop reduce average hinge loss.

In [ ]:
losses_verified_w = np.array([0.235, 0.070, 0.505])
R_S_w = float(np.mean(losses_verified_w))
cost_w = 0.070
score_w = R_S_w + cost_w
print("empirical risk:", round(R_S_w, 3), "cost:", round(cost_w, 3), "decision score:", round(score_w, 3))
assert round(R_S_w, 3) == 0.270
assert round(score_w, 3) == 0.340

▶ What you'll see: the raw training average is not the final decision score; the cost term changes the number used for selection.

In [ ]:
X_stream_w = np.array([[2.0, 1.0], [1.0, 2.0], [-2.0, -1.0], [-1.0, -2.0], [1.5, 0.5], [-0.5, -1.5]])
y_stream_w = np.array([1.0, 1.0, -1.0, -1.0, 1.0, -1.0])
w_stream_w = np.zeros(2)
risk_curve_w = []
for epoch_w in range(8):
    margins_epoch_w = y_stream_w * (X_stream_w @ w_stream_w)
    risk_curve_w.append(float(np.mean(np.maximum(0.0, 1.0 - margins_epoch_w))))
    for x_i_w, y_i_w in zip(X_stream_w, y_stream_w):
        loss_i_w = max(0.0, 1.0 - y_i_w * float(w_stream_w @ x_i_w))
        tau_i_w = loss_i_w / float(x_i_w @ x_i_w) if loss_i_w > 0 else 0.0
        w_stream_w = w_stream_w + tau_i_w * y_i_w * x_i_w
print("risk start -> end:", round(risk_curve_w[0], 3), "->", round(risk_curve_w[-1], 3))
print("trained w:", np.round(w_stream_w, 3))
assert risk_curve_w[-1] < risk_curve_w[0]

▶ What you'll see: average hinge loss falls as the stream updates the margin-violating examples.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.plot(risk_curve_w, marker="o", color="purple")
plt.title("5: empirical hinge risk during PA training")
plt.xlabel("epoch"); plt.ylabel("average hinge loss")
plt.show()

▶ What you'll see: a quick drop then a plateau once the small stream is safely separated.

*Why it's done this way:* empirical risk is the average of per-example losses, so it is the right aggregate to inspect during training. Adding cost or regularization prevents selecting a model just because it made the training curve pretty.

### 6. Comparing baseline, flexible, and stabilized decisions

The content block's final arithmetic compares a baseline score `0.340`, a tempting flexible alternative `0.384`, and a stabilized score `0.272`. Lower is better on this decision scale. The gap and relative gap quantify whether the flexible alternative has enough evidence to justify its extra complexity.

In [ ]:
baseline_w = 0.340
flexible_w = 0.384
stable_w = 0.80 * baseline_w
gap_w = flexible_w - baseline_w
relative_gap_w = gap_w / flexible_w
scores_compare_w = np.array([baseline_w, flexible_w, stable_w])
print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))
print("stable score:", round(stable_w, 3), "winner score:", round(float(np.min(scores_compare_w)), 3))
assert round(gap_w, 3) == 0.044
assert round(relative_gap_w, 3) == 0.115
assert round(stable_w, 3) == 0.272

▶ What you'll see: the stabilized score is lowest, while the flexible alternative is worse by an 11.5% relative gap.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(["baseline", "flexible", "stabilized"], scores_compare_w, color=["gray", "crimson", "teal"])
plt.title("6: choose by full decision score")
plt.ylabel("score (lower is better)")
plt.show()

▶ What you'll see: the lowest bar is the stabilized model, not the flexible alternative.

*Why it's done this way:* PA itself is a margin rule, but choosing a usable learner still requires the full score implied by the method. A raw fit, a penalty or cost, and a validation-relevant gap are different pieces of evidence; mixing them only after putting them on the same scale is what makes the comparison meaningful.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for vectors, dot products, norms, masks, and deterministic online streams.
import matplotlib.pyplot as plt # load Matplotlib so each margin, update, and score comparison can be inspected visually.
np.random.seed(0) # make all stochastic examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Compute a signed margin

**Goal.** Convert a score into a signed margin, because PA cares whether the label is correct and safely separated. We build it in 2 steps.

In [ ]:
x_b1 = np.array([2.0, 1.0]) # define one feature vector for a binary classifier.
y_b1 = 1.0 # encode the positive class as +1.
w_b1 = np.array([0.2, -0.1]) # choose a weak current weight vector.
print("x:", x_b1, "y:", y_b1, "w:", w_b1) # inspect the ingredients before scoring.

In [ ]:
score_b1 = float(w_b1 @ x_b1) # compute the raw linear decision score.
margin_b1 = y_b1 * score_b1 # multiply by the label so correct confident examples are large positive numbers.
print("score:", round(score_b1, 3), "signed margin:", round(margin_b1, 3)) # inspect the PA margin.
assert round(margin_b1, 3) == 0.300 # verify 1*(0.2*2 - 0.1*1).
plt.figure(figsize=(4, 3)) # create a compact diagnostic plot.
plt.bar(["score", "y·score", "target margin"], [score_b1, margin_b1, 1.0], color=["gray", "teal", "black"]) # compare the current margin with the safety target.
plt.title("Basic 1: signed margin") # title the plot.
plt.show() # display the bars.

▶ What you'll see: the example is on the correct side but far below the target margin of 1.

👀 Takeaway: PA's unit of correctness is `y wᵀx`, not the raw score alone.

### Basic 2 — Turn margin debt into hinge loss

**Goal.** Compute `max(0, 1 - margin)`, because that is the per-example violation PA reacts to. We build it in 2 steps.

In [ ]:
margin_b2 = 0.30 # reuse a small positive margin from the previous worked example.
loss_b2 = max(0.0, 1.0 - margin_b2) # compute hinge loss as the missing amount up to margin 1.
print("margin:", margin_b2, "hinge loss:", round(loss_b2, 3)) # inspect the violation.
assert round(loss_b2, 3) == 0.700 # verify the margin debt.

In [ ]:
m_grid_b2 = np.linspace(-1.0, 2.0, 80) # create possible signed margins.
hinge_b2 = np.maximum(0.0, 1.0 - m_grid_b2) # compute hinge loss for each margin.
plt.figure(figsize=(4, 3)) # create a compact curve.
plt.plot(m_grid_b2, hinge_b2, color="purple") # draw the hinge loss shape.
plt.scatter([margin_b2], [loss_b2], color="red") # mark the current example.
plt.axvline(1.0, color="black", linestyle="--") # mark the safe margin boundary.
plt.title("Basic 2: hinge loss") # title the plot.
plt.xlabel("margin"); plt.ylabel("loss") # label axes.
plt.show() # display the plot.

▶ What you'll see: loss decreases linearly until margin 1, then stays at zero.

👀 Takeaway: PA updates only when the hinge loss is positive.

### Basic 3 — Compute the PA step size

**Goal.** Divide loss by `||x||²`, because a step along `y x` changes the margin in proportion to the squared norm of `x`. We build it in 2 steps.

In [ ]:
x_b3 = np.array([2.0, 1.0]) # define the same example vector.
loss_b3 = 0.70 # define the hinge violation to repair.
norm_sq_b3 = float(x_b3 @ x_b3) # compute ||x||^2.
tau_b3 = loss_b3 / norm_sq_b3 # compute the exact PA step size.
print("||x||^2:", round(norm_sq_b3, 3), "tau:", round(tau_b3, 3)) # inspect the denominator and step.
assert round(tau_b3, 3) == 0.140 # verify 0.70 / 5.

In [ ]:
plt.figure(figsize=(4, 3)) # create a small comparison chart.
plt.bar(["loss", "||x||²", "τ"], [loss_b3, norm_sq_b3, tau_b3], color=["crimson", "gray", "teal"]) # show the scale conversion.
plt.title("Basic 3: loss divided by squared norm") # title the chart.
plt.show() # display the bars.

▶ What you'll see: a 0.7 margin debt becomes a much smaller coordinate step because `||x||² = 5`.

👀 Takeaway: larger feature vectors need smaller τ to produce the same margin change.

### Basic 4 — Apply one PA update

**Goal.** Update `w` by `τ y x`, because PA corrects the current example in the label's direction. We build it in 3 steps.

In [ ]:
w_b4 = np.array([0.2, -0.1]) # current weights before correction.
x_b4 = np.array([2.0, 1.0]) # current example.
y_b4 = 1.0 # positive class label.
tau_b4 = 0.14 # step size from Basic 3.
print("before w:", w_b4) # inspect the starting point.

In [ ]:
w_new_b4 = w_b4 + tau_b4 * y_b4 * x_b4 # apply the passive-aggressive update rule.
margin_after_b4 = float(y_b4 * (w_new_b4 @ x_b4)) # check the post-update margin.
print("after w:", np.round(w_new_b4, 3), "new margin:", round(margin_after_b4, 3)) # inspect the correction.
assert np.allclose(np.round(w_new_b4, 3), [0.48, 0.04]) # verify the updated weights.
assert round(margin_after_b4, 3) == 1.000 # verify the example is exactly repaired.

In [ ]:
plt.figure(figsize=(4, 3)) # create a before-after chart.
plt.bar(["old margin", "new margin", "target"], [0.3, margin_after_b4, 1.0], color=["gray", "teal", "black"]) # show the margin repair.
plt.title("Basic 4: one PA update") # title the chart.
plt.show() # display the chart.

▶ What you'll see: one update lifts the margin exactly to the target value 1.

👀 Takeaway: the uncapped PA step is the smallest update that fixes the current example.

### Basic 5 — Stay passive on a safe example

**Goal.** Show that a safe margin gives zero update, because PA should not move weights unnecessarily. We build it in 2 steps.

In [ ]:
w_b5 = np.array([0.48, 0.04]) # a weight vector that already repaired x=(2,1).
x_b5 = np.array([2.0, 1.0]) # the same positive example.
y_b5 = 1.0 # positive class.
margin_b5 = float(y_b5 * (w_b5 @ x_b5)) # compute the signed margin.
loss_b5 = max(0.0, 1.0 - margin_b5) # compute hinge loss.
print("margin:", round(margin_b5, 3), "loss:", round(loss_b5, 3)) # inspect the no-update condition.
assert round(loss_b5, 3) == 0.000 # verify safety.

In [ ]:
tau_b5 = loss_b5 / float(x_b5 @ x_b5) if loss_b5 > 0 else 0.0 # return zero when no violation exists.
w_same_b5 = w_b5 + tau_b5 * y_b5 * x_b5 # apply the update rule.
print("tau:", tau_b5, "weights unchanged:", np.allclose(w_same_b5, w_b5)) # verify passivity.
plt.figure(figsize=(4, 3)) # create a compact plot.
plt.bar(["old w0", "new w0", "old w1", "new w1"], [w_b5[0], w_same_b5[0], w_b5[1], w_same_b5[1]], color="teal") # compare coordinates.
plt.title("Basic 5: passive when safe") # title the chart.
plt.xticks(rotation=20) # fit labels.
plt.show() # display the bars.

▶ What you'll see: τ is zero and the old and new weights are identical.

👀 Takeaway: PA is aggressive only on violations and passive otherwise.

### Basic 6 — Update on a negative example

**Goal.** Use a label `-1`, because the same formula must push negative examples to the opposite side. We build it in 3 steps.

In [ ]:
w_b6 = np.array([0.1, 0.1]) # start with weights that mistakenly score a negative point positively.
x_b6 = np.array([1.0, 2.0]) # current example.
y_b6 = -1.0 # negative class label.
margin_b6 = float(y_b6 * (w_b6 @ x_b6)) # compute signed margin.
loss_b6 = max(0.0, 1.0 - margin_b6) # compute hinge violation.
print("margin:", round(margin_b6, 3), "loss:", round(loss_b6, 3)) # inspect the mistake.
assert round(loss_b6, 3) == 1.300 # verify the negative example is badly violated.

In [ ]:
tau_b6 = loss_b6 / float(x_b6 @ x_b6) # compute the PA step size.
w_new_b6 = w_b6 + tau_b6 * y_b6 * x_b6 # y=-1 pushes weights opposite x.
new_margin_b6 = float(y_b6 * (w_new_b6 @ x_b6)) # verify repaired margin.
print("tau:", round(tau_b6, 3), "new w:", np.round(w_new_b6, 3), "new margin:", round(new_margin_b6, 3)) # inspect the correction.
assert round(tau_b6, 3) == 0.260 # verify 1.3 / 5.
assert round(new_margin_b6, 3) == 1.000 # verify exact repair.

In [ ]:
plt.figure(figsize=(4, 3)) # create a before-after margin plot.
plt.bar(["before", "after", "target"], [margin_b6, new_margin_b6, 1.0], color=["crimson", "teal", "black"]) # show the correction.
plt.title("Basic 6: negative-label update") # title the plot.
plt.ylabel("signed margin") # label the scale.
plt.show() # display the plot.

▶ What you'll see: the negative point moves from a bad margin to a safe margin.

👀 Takeaway: multiplying by `y` gives one formula for both classes.

### Basic 7 — Cap a too-large PA-I step

**Goal.** Apply `min(C, τ)`, because noisy small-norm examples can demand huge raw updates. We build it in 2 steps.

In [ ]:
loss_b7 = 1.3 # define a large violation.
norm_sq_b7 = 0.05 # define a tiny feature norm squared.
raw_tau_b7 = loss_b7 / norm_sq_b7 # compute the uncapped step.
C_b7 = 2.0 # define the PA-I aggressiveness limit.
tau_b7 = min(C_b7, raw_tau_b7) # cap the step.
print("raw tau:", round(raw_tau_b7, 3), "capped tau:", round(tau_b7, 3)) # inspect stabilization.
assert round(raw_tau_b7, 3) == 26.000 # verify raw step.
assert round(tau_b7, 3) == 2.000 # verify cap.

In [ ]:
plt.figure(figsize=(4, 3)) # create a cap comparison chart.
plt.bar(["raw τ", "C", "PA-I τ"], [raw_tau_b7, C_b7, tau_b7], color=["crimson", "gray", "teal"]) # compare raw and capped values.
plt.title("Basic 7: PA-I cap") # title the plot.
plt.ylabel("step size") # label the scale.
plt.show() # display the plot.

▶ What you'll see: the raw step towers over the cap, so PA-I uses C instead.

👀 Takeaway: `C` is a stability knob that limits how hard one example can move the model.

### Basic 8 — Compute a PA-II softened step

**Goal.** Use `loss / (||x||² + 1/(2C))`, because PA-II shrinks extreme updates smoothly rather than with a hard cap. We build it in 2 steps.

In [ ]:
loss_b8 = 1.3 # define the same violation.
norm_sq_b8 = 0.05 # define the same tiny squared norm.
C_b8 = 2.0 # define the softness parameter.
tau_b8 = loss_b8 / (norm_sq_b8 + 1.0 / (2.0 * C_b8)) # compute PA-II update size.
print("PA-II tau:", round(tau_b8, 3)) # inspect the softened step.
assert round(tau_b8, 3) == 4.333 # verify 1.3 / (0.05 + 0.25).

In [ ]:
C_grid_b8 = np.array([0.5, 1.0, 2.0, 10.0]) # try increasing aggressiveness.
taus_b8 = loss_b8 / (norm_sq_b8 + 1.0 / (2.0 * C_grid_b8)) # compute PA-II steps for each C.
print("taus by C:", np.round(taus_b8, 3)) # inspect how C changes update size.
plt.figure(figsize=(4, 3)) # create a curve.
plt.plot(C_grid_b8, taus_b8, marker="o", color="orange") # plot softness versus C.
plt.title("Basic 8: PA-II softness") # title the plot.
plt.xlabel("C"); plt.ylabel("τ") # label axes.
plt.show() # display the line chart.

▶ What you'll see: larger C makes PA-II more aggressive and closer to raw PA.

👀 Takeaway: PA-II trades exact repair for smoother, regularized movement.

### Basic 9 — Average verified training losses

**Goal.** Recompute the lesson's empirical risk, because PA model selection starts from an average per-example loss. We build it in 2 steps.

In [ ]:
losses_b9 = np.array([0.235, 0.070, 0.505]) # use the verified toy losses from the lesson content.
R_S_b9 = float(np.mean(losses_b9)) # compute empirical risk as the average loss.
print("losses:", losses_b9, "R_S:", round(R_S_b9, 3)) # inspect the average.
assert round(R_S_b9, 3) == 0.270 # verify (0.235+0.070+0.505)/3.

In [ ]:
plt.figure(figsize=(4, 3)) # create a per-example loss chart.
plt.bar(["ex1", "ex2", "ex3"], losses_b9, color="purple") # show individual losses.
plt.axhline(R_S_b9, color="black", linestyle="--", label="average") # show empirical risk.
plt.title("Basic 9: empirical risk") # title the chart.
plt.legend() # show average label.
plt.show() # display the chart.

▶ What you'll see: the average line sits at 0.270 across the three verified losses.

👀 Takeaway: empirical risk is an average, not the best or worst single example.

### Basic 10 — Add the method cost

**Goal.** Add the complexity or operational cost to the raw empirical risk, because selection should use the full decision score. We build it in 2 steps.

In [ ]:
R_S_b10 = 0.270 # verified empirical risk.
cost_b10 = 0.070 # verified cost term for this lesson.
score_b10 = R_S_b10 + cost_b10 # compute the full selection score.
print("R_S:", R_S_b10, "cost:", cost_b10, "score:", round(score_b10, 3)) # inspect the full score.
assert round(score_b10, 3) == 0.340 # verify the lesson score.

In [ ]:
plt.figure(figsize=(4, 3)) # create an additive score chart.
plt.bar(["empirical risk", "cost", "total score"], [R_S_b10, cost_b10, score_b10], color=["teal", "orange", "gray"]) # show how the score is assembled.
plt.title("Basic 10: raw fit plus cost") # title the chart.
plt.xticks(rotation=15) # make labels readable.
plt.show() # display the chart.

▶ What you'll see: the final score is higher than the raw average because the cost is part of the decision.

👀 Takeaway: dropping the cost term changes the algorithm's selection criterion.

## 🟡 Easy

### Easy 1 — Train a tiny PA classifier

**Goal.** Run several online passes, because PA learns from a stream of margin violations rather than from one batch solve. We build it in 4 steps.

In [ ]:
X_e1 = np.array([[2.0, 1.0], [1.0, 2.0], [-2.0, -1.0], [-1.0, -2.0], [1.5, 0.5], [-0.5, -1.5]]) # define a separable stream.
y_e1 = np.array([1.0, 1.0, -1.0, -1.0, 1.0, -1.0]) # labels as ±1.
w_e1 = np.zeros(2) # start from no preference.
print("initial w:", w_e1) # inspect initialization.

In [ ]:
loss_curve_e1 = [] # store average hinge loss per epoch.
updates_e1 = 0 # count nonzero PA updates.
for epoch_e1 in range(8): # make repeated passes through the stream.
    margins_e1 = y_e1 * (X_e1 @ w_e1) # compute margins before this epoch's updates.
    loss_curve_e1.append(float(np.mean(np.maximum(0.0, 1.0 - margins_e1)))) # record empirical risk.
    for x_i_e1, y_i_e1 in zip(X_e1, y_e1): # process one example at a time.
        loss_i_e1 = max(0.0, 1.0 - y_i_e1 * float(w_e1 @ x_i_e1)) # compute hinge loss for this example.
        if loss_i_e1 > 0: # update only violations.
            tau_i_e1 = loss_i_e1 / float(x_i_e1 @ x_i_e1) # exact PA step.
            w_e1 = w_e1 + tau_i_e1 * y_i_e1 * x_i_e1 # apply update.
            updates_e1 += 1 # count the movement.
print("updates:", updates_e1, "final w:", np.round(w_e1, 3)) # inspect training result.

In [ ]:
final_margins_e1 = y_e1 * (X_e1 @ w_e1) # compute final margins.
final_loss_e1 = float(np.mean(np.maximum(0.0, 1.0 - final_margins_e1))) # compute final empirical hinge risk.
print("loss start -> final:", round(loss_curve_e1[0], 3), "->", round(final_loss_e1, 3)) # inspect improvement.
assert final_loss_e1 < loss_curve_e1[0] # verify training improved risk.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create a learning curve plot.
plt.plot(loss_curve_e1 + [final_loss_e1], marker="o", color="teal") # plot risk over passes.
plt.title("Easy 1: PA training curve") # title the plot.
plt.xlabel("epoch"); plt.ylabel("average hinge loss") # label axes.
plt.show() # display the curve.

▶ What you'll see: the average hinge loss falls quickly as violating examples trigger updates.

👀 Takeaway: PA is an online learner whose training loop is margin-check, update-if-needed, repeat.

### Easy 2 — Plot the learned separator

**Goal.** Visualize the classifier after PA training, because the margin rule is geometric. We build it in 4 steps.

In [ ]:
X_e2 = np.array([[2.0, 1.0], [1.0, 2.0], [-2.0, -1.0], [-1.0, -2.0], [1.5, 0.5], [-0.5, -1.5]]) # define the same stream.
y_e2 = np.array([1.0, 1.0, -1.0, -1.0, 1.0, -1.0]) # labels.
w_e2 = np.zeros(2) # initialize weights.
print("points:", X_e2.shape[0]) # inspect stream size.

In [ ]:
for epoch_e2 in range(6): # train for a few passes.
    for x_i_e2, y_i_e2 in zip(X_e2, y_e2): # process online examples.
        loss_i_e2 = max(0.0, 1.0 - y_i_e2 * float(w_e2 @ x_i_e2)) # compute violation.
        tau_i_e2 = loss_i_e2 / float(x_i_e2 @ x_i_e2) if loss_i_e2 > 0 else 0.0 # compute PA step or zero.
        w_e2 = w_e2 + tau_i_e2 * y_i_e2 * x_i_e2 # update weights.
print("learned w:", np.round(w_e2, 3)) # inspect learned separator.

In [ ]:
pred_e2 = np.sign(X_e2 @ w_e2) # classify training points by sign.
acc_e2 = float(np.mean(pred_e2 == y_e2)) # compute simple training accuracy.
min_margin_e2 = float(np.min(y_e2 * (X_e2 @ w_e2))) # inspect worst margin.
print("accuracy:", acc_e2, "minimum margin:", round(min_margin_e2, 3)) # inspect correctness and safety.
assert acc_e2 == 1.0 # verify the toy stream is classified correctly.

In [ ]:
plt.figure(figsize=(4.5, 3.5)) # create the separator plot.
plt.scatter(X_e2[y_e2 == 1, 0], X_e2[y_e2 == 1, 1], color="teal", label="+1") # plot positives.
plt.scatter(X_e2[y_e2 == -1, 0], X_e2[y_e2 == -1, 1], color="crimson", label="-1") # plot negatives.
grid_e2 = np.linspace(-2.5, 2.5, 100) # x-axis grid for the line.
plt.plot(grid_e2, -(w_e2[0] / w_e2[1]) * grid_e2, color="black", label="w·x=0") # draw decision boundary.
plt.title("Easy 2: learned PA separator") # title the plot.
plt.legend(); plt.show() # display plot.

▶ What you'll see: the line separates the two small clusters after online updates.

👀 Takeaway: PA's algebraic margin updates produce a visible linear boundary.

### Easy 3 — Compare raw PA and PA-I on a noisy point

**Goal.** Show why capping can help, because one suspicious example can dominate the uncapped update. We build it in 4 steps.

In [ ]:
w_e3 = np.array([-1.0, -1.0]) # current weights before seeing a tiny-norm positive point.
x_e3 = np.array([0.2, 0.1]) # tiny feature vector that creates a large raw τ.
y_e3 = 1.0 # positive label.
C_e3 = 2.0 # PA-I cap.
loss_e3 = max(0.0, 1.0 - y_e3 * float(w_e3 @ x_e3)) # compute violation.
print("loss:", round(loss_e3, 3)) # inspect margin debt.

In [ ]:
raw_tau_e3 = loss_e3 / float(x_e3 @ x_e3) # compute uncapped PA step.
cap_tau_e3 = min(C_e3, raw_tau_e3) # compute PA-I capped step.
w_raw_e3 = w_e3 + raw_tau_e3 * y_e3 * x_e3 # apply raw PA.
w_cap_e3 = w_e3 + cap_tau_e3 * y_e3 * x_e3 # apply capped PA-I.
print("raw tau:", round(raw_tau_e3, 3), "cap tau:", round(cap_tau_e3, 3)) # inspect step sizes.
assert round(raw_tau_e3, 3) == 26.000 # verify raw step.

In [ ]:
move_raw_e3 = float(np.linalg.norm(w_raw_e3 - w_e3)) # measure raw movement.
move_cap_e3 = float(np.linalg.norm(w_cap_e3 - w_e3)) # measure capped movement.
print("movement raw:", round(move_raw_e3, 3), "movement capped:", round(move_cap_e3, 3)) # compare update sizes.
assert move_cap_e3 < move_raw_e3 # verify stabilization.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create a movement comparison.
plt.bar(["raw PA", "PA-I capped"], [move_raw_e3, move_cap_e3], color=["crimson", "teal"]) # compare norm of weight movement.
plt.title("Easy 3: capping reduces one-point dominance") # title the plot.
plt.ylabel("||Δw||") # label the movement scale.
plt.show() # display chart.

▶ What you'll see: the capped update moves far less than the raw exact-repair update.

👀 Takeaway: PA-I keeps the online learner from overreacting to one high-leverage example.

### Easy 4 — Compute the verified score and gap

**Goal.** Recreate the lesson's score comparison, because raw training fit must be judged against cost and alternatives. We build it in 3 steps.

In [ ]:
losses_e4 = np.array([0.235, 0.070, 0.505]) # verified per-example toy losses.
cost_e4 = 0.070 # verified cost term.
flexible_e4 = 0.384 # verified score for a more flexible alternative.
R_S_e4 = float(np.mean(losses_e4)) # compute empirical risk.
score_e4 = R_S_e4 + cost_e4 # compute baseline decision score.
print("R_S:", round(R_S_e4, 3), "score:", round(score_e4, 3)) # inspect score construction.
assert round(score_e4, 3) == 0.340 # verify content arithmetic.

In [ ]:
gap_e4 = flexible_e4 - score_e4 # compute absolute gap.
relative_gap_e4 = gap_e4 / flexible_e4 # compute relative gap against the alternative scale.
print("gap:", round(gap_e4, 3), "relative gap:", round(relative_gap_e4, 3)) # inspect evidence size.
assert round(gap_e4, 3) == 0.044 # verify absolute gap.
assert round(relative_gap_e4, 3) == 0.115 # verify relative gap.

In [ ]:
plt.figure(figsize=(4, 3)) # create a score comparison chart.
plt.bar(["baseline score", "flexible score"], [score_e4, flexible_e4], color=["teal", "crimson"]) # compare options.
plt.title("Easy 4: full-score gap") # title the chart.
plt.ylabel("score, lower is better") # label the scale.
plt.show() # display chart.

▶ What you'll see: the flexible model must overcome a 0.044 absolute gap, about 11.5% of its score.

👀 Takeaway: a lower raw term is not enough; compare full scores on one scale.

### Easy 5 — Apply a stabilization multiplier

**Goal.** Compute the stabilized decision score, because the lesson's final choice rewards the safer option. We build it in 3 steps.

In [ ]:
baseline_e5 = 0.340 # verified baseline score.
stabilizer_e5 = 0.80 # verified 20% reduction multiplier.
stable_e5 = stabilizer_e5 * baseline_e5 # compute stabilized score.
print("stable score:", round(stable_e5, 3)) # inspect stabilized option.
assert round(stable_e5, 3) == 0.272 # verify 0.80*0.340.

In [ ]:
scores_e5 = np.array([baseline_e5, 0.384, stable_e5]) # collect baseline, flexible, and stable scores.
names_e5 = np.array(["baseline", "flexible", "stabilized"]) # name each option.
winner_e5 = names_e5[int(np.argmin(scores_e5))] # choose the lowest full score.
print("scores:", np.round(scores_e5, 3), "winner:", winner_e5) # inspect final decision.
assert winner_e5 == "stabilized" # verify content conclusion.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create final decision chart.
plt.bar(names_e5, scores_e5, color=["gray", "crimson", "teal"]) # compare final options.
plt.title("Easy 5: stabilized option wins") # title plot.
plt.ylabel("decision score") # label scale.
plt.show() # display chart.

▶ What you'll see: the stabilized option has the lowest score among the three.

👀 Takeaway: selection uses the complete score implied by the method, not the prettiest training fragment.

## 🔴 Advanced

### Advanced 1 — Compare PA, PA-I, and PA-II learning curves

**Goal.** Train three variants on the same stream, because different aggressiveness rules change stability. We build it in 5 steps.

In [ ]:
X_a1 = np.array([[2.0, 1.0], [1.0, 2.0], [-2.0, -1.0], [-1.0, -2.0], [0.2, 0.1], [-0.2, -0.1]]) # include two tiny-norm edge cases.
y_a1 = np.array([1.0, 1.0, -1.0, -1.0, -1.0, 1.0]) # make tiny points conflict with the large-cluster pattern.
variants_a1 = ["PA", "PA-I", "PA-II"] # compare three update rules.
C_a1 = 1.0 # use the same aggressiveness parameter for stabilized variants.
print("variants:", variants_a1) # inspect experiment settings.

In [ ]:
curves_a1 = [] # store one loss curve per variant.
weights_a1 = [] # store final weights.
for name_a1 in variants_a1: # train each variant from scratch.
    w_a1 = np.zeros(2) # reset weights.
    curve_a1 = [] # record average hinge loss.
    for epoch_a1 in range(12): # repeated online passes.
        margins_a1 = y_a1 * (X_a1 @ w_a1) # compute current margins.
        curve_a1.append(float(np.mean(np.maximum(0.0, 1.0 - margins_a1)))) # record empirical risk.
        for x_i_a1, y_i_a1 in zip(X_a1, y_a1): # scan stream.
            loss_i_a1 = max(0.0, 1.0 - y_i_a1 * float(w_a1 @ x_i_a1)) # current violation.
            norm_i_a1 = float(x_i_a1 @ x_i_a1) # squared norm.
            if loss_i_a1 == 0.0: # no violation.
                tau_i_a1 = 0.0 # stay passive.
            elif name_a1 == "PA": # exact-repair PA.
                tau_i_a1 = loss_i_a1 / norm_i_a1 # raw step.
            elif name_a1 == "PA-I": # capped PA.
                tau_i_a1 = min(C_a1, loss_i_a1 / norm_i_a1) # hard cap.
            else: # PA-II.
                tau_i_a1 = loss_i_a1 / (norm_i_a1 + 1.0 / (2.0 * C_a1)) # soft denominator.
            w_a1 = w_a1 + tau_i_a1 * y_i_a1 * x_i_a1 # apply selected update.
    curves_a1.append(curve_a1) # save this curve.
    weights_a1.append(w_a1) # save final weights.
print("final risks:", [round(c[-1], 3) for c in curves_a1]) # inspect final losses.

In [ ]:
norms_a1 = np.array([np.linalg.norm(w) for w in weights_a1]) # compute final weight norms.
print("weight norms:", np.round(norms_a1, 3)) # inspect movement scale.
assert np.all(norms_a1 >= 0) # concrete sanity check that all variants produced finite weights.

In [ ]:
plt.figure(figsize=(5, 3)) # create a learning-curve comparison.
for name_a1, curve_a1 in zip(variants_a1, curves_a1): # plot each variant.
    plt.plot(curve_a1, marker="o", label=name_a1) # draw one curve.
plt.title("Advanced 1: PA variant loss curves") # title plot.
plt.xlabel("epoch"); plt.ylabel("average hinge loss") # label axes.
plt.legend(); plt.show() # display chart.

▶ What you'll see: stabilized variants usually move less erratically on the conflicting tiny-norm examples.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create a norm comparison.
plt.bar(variants_a1, norms_a1, color=["gray", "teal", "orange"]) # compare model sizes.
plt.title("Advanced 1: final weight norms") # title chart.
plt.ylabel("||w||") # label scale.
plt.show() # display chart.

▶ What you'll see: the capped and softened variants restrain weight movement relative to aggressive exact repair.

👀 Takeaway: PA variants are different answers to the same question: how much should one violation be allowed to change the model?

### Advanced 2 — Hold out examples for validation

**Goal.** Compare training and validation hinge loss, because an online learner should be judged on future-like examples, not just the stream it saw. We build it in 5 steps.

In [ ]:
X_train_a2 = np.array([[2.0, 1.0], [1.0, 2.0], [-2.0, -1.0], [-1.0, -2.0], [1.5, 0.5], [-0.5, -1.5]]) # training stream.
y_train_a2 = np.array([1.0, 1.0, -1.0, -1.0, 1.0, -1.0]) # training labels.
X_val_a2 = np.array([[1.2, 1.0], [-1.2, -1.0], [0.4, 0.8], [-0.8, -0.4]]) # held-out examples.
y_val_a2 = np.array([1.0, -1.0, 1.0, -1.0]) # held-out labels.
print("train/val sizes:", len(y_train_a2), len(y_val_a2)) # inspect split sizes.

In [ ]:
w_a2 = np.zeros(2) # initialize PA weights.
train_curve_a2 = [] # store training hinge loss.
val_curve_a2 = [] # store validation hinge loss.
for epoch_a2 in range(10): # train over multiple epochs.
    train_curve_a2.append(float(np.mean(np.maximum(0.0, 1.0 - y_train_a2 * (X_train_a2 @ w_a2))))) # record train loss.
    val_curve_a2.append(float(np.mean(np.maximum(0.0, 1.0 - y_val_a2 * (X_val_a2 @ w_a2))))) # record val loss before updates.
    for x_i_a2, y_i_a2 in zip(X_train_a2, y_train_a2): # scan the training stream.
        loss_i_a2 = max(0.0, 1.0 - y_i_a2 * float(w_a2 @ x_i_a2)) # compute violation.
        tau_i_a2 = loss_i_a2 / float(x_i_a2 @ x_i_a2) if loss_i_a2 > 0 else 0.0 # PA step.
        w_a2 = w_a2 + tau_i_a2 * y_i_a2 * x_i_a2 # update.
print("final w:", np.round(w_a2, 3)) # inspect trained model.

In [ ]:
train_final_a2 = float(np.mean(np.maximum(0.0, 1.0 - y_train_a2 * (X_train_a2 @ w_a2)))) # final train risk.
val_final_a2 = float(np.mean(np.maximum(0.0, 1.0 - y_val_a2 * (X_val_a2 @ w_a2)))) # final validation risk.
print("train final:", round(train_final_a2, 3), "val final:", round(val_final_a2, 3)) # inspect both risks.
assert val_final_a2 >= 0.0 # verify valid nonnegative hinge loss.

In [ ]:
plt.figure(figsize=(5, 3)) # create train/validation plot.
plt.plot(train_curve_a2 + [train_final_a2], marker="o", label="train") # plot train curve.
plt.plot(val_curve_a2 + [val_final_a2], marker="s", label="validation") # plot validation curve.
plt.title("Advanced 2: train versus validation hinge loss") # title plot.
plt.xlabel("epoch"); plt.ylabel("average hinge loss") # label axes.
plt.legend(); plt.show() # display chart.

▶ What you'll see: validation follows the same broad improvement but is the number that checks future behavior.

In [ ]:
scores_a2 = np.array([train_final_a2, val_final_a2]) # collect final risks.
plt.figure(figsize=(4, 3)) # create final comparison chart.
plt.bar(["train", "validation"], scores_a2, color=["teal", "orange"]) # compare final losses.
plt.title("Advanced 2: final split losses") # title chart.
plt.ylabel("hinge loss") # label scale.
plt.show() # display chart.

▶ What you'll see: the split view makes clear whether the training improvement survived held-out data.

👀 Takeaway: PA updates minimize stream violations, but model selection should still look at validation behavior.

### Advanced 3 — Sweep C and choose by validation score

**Goal.** Tune the PA-I cap, because too much aggressiveness may overreact and too little may underfit. We build it in 5 steps.

In [ ]:
X_a3 = np.array([[2.0, 1.0], [1.0, 2.0], [-2.0, -1.0], [-1.0, -2.0], [0.2, 0.1], [-0.2, -0.1]]) # training stream with conflicting tiny points.
y_a3 = np.array([1.0, 1.0, -1.0, -1.0, -1.0, 1.0]) # labels include edge conflicts.
X_val_a3 = np.array([[1.5, 1.0], [-1.5, -1.0], [0.3, 0.2], [-0.3, -0.2]]) # validation points.
y_val_a3 = np.array([1.0, -1.0, 1.0, -1.0]) # validation labels favor the main pattern.
Cs_a3 = np.array([0.1, 0.5, 1.0, 5.0]) # cap values to compare.
print("C grid:", Cs_a3) # inspect sweep.

In [ ]:
train_losses_a3 = [] # store final training losses.
val_losses_a3 = [] # store final validation losses.
for C_a3 in Cs_a3: # train one PA-I model per cap.
    w_a3 = np.zeros(2) # reset weights.
    for epoch_a3 in range(14): # repeat stream.
        for x_i_a3, y_i_a3 in zip(X_a3, y_a3): # scan examples.
            loss_i_a3 = max(0.0, 1.0 - y_i_a3 * float(w_a3 @ x_i_a3)) # compute violation.
            tau_i_a3 = min(C_a3, loss_i_a3 / float(x_i_a3 @ x_i_a3)) if loss_i_a3 > 0 else 0.0 # PA-I step.
            w_a3 = w_a3 + tau_i_a3 * y_i_a3 * x_i_a3 # update.
    train_losses_a3.append(float(np.mean(np.maximum(0.0, 1.0 - y_a3 * (X_a3 @ w_a3))))) # record train hinge.
    val_losses_a3.append(float(np.mean(np.maximum(0.0, 1.0 - y_val_a3 * (X_val_a3 @ w_a3))))) # record val hinge.
print("train losses:", np.round(train_losses_a3, 3)) # inspect training sweep.
print("val losses:", np.round(val_losses_a3, 3)) # inspect validation sweep.

In [ ]:
best_idx_a3 = int(np.argmin(val_losses_a3)) # choose cap by validation loss.
best_C_a3 = float(Cs_a3[best_idx_a3]) # read best cap.
print("best C:", best_C_a3, "best val loss:", round(val_losses_a3[best_idx_a3], 3)) # inspect selected cap.
assert best_C_a3 in Cs_a3 # verify selected value came from the grid.

In [ ]:
plt.figure(figsize=(5, 3)) # create C-sweep plot.
plt.plot(Cs_a3, train_losses_a3, marker="o", label="train") # plot training loss.
plt.plot(Cs_a3, val_losses_a3, marker="s", label="validation") # plot validation loss.
plt.axvline(best_C_a3, color="red", linestyle="--", label="best C") # mark selected cap.
plt.title("Advanced 3: choose C by validation") # title plot.
plt.xlabel("C"); plt.ylabel("hinge loss") # label axes.
plt.legend(); plt.show() # display chart.

▶ What you'll see: the best cap is the one with the lowest validation curve, not necessarily the lowest training curve.

In [ ]:
plt.figure(figsize=(4, 3)) # create final validation bar chart.
plt.bar([str(c) for c in Cs_a3], val_losses_a3, color="teal") # compare validation scores.
plt.title("Advanced 3: validation scores by C") # title chart.
plt.xlabel("C"); plt.ylabel("validation hinge") # label axes.
plt.show() # display chart.

▶ What you'll see: the validation bars make the selected cap visible.

👀 Takeaway: `C` is a regularization knob, so it belongs in a validation sweep.

### Advanced 4 — Use averaged PA weights

**Goal.** Compare final weights with averaged weights, because averaging online iterates often stabilizes noisy update paths. We build it in 5 steps.

In [ ]:
X_a4 = np.array([[2.0, 1.0], [1.0, 2.0], [-2.0, -1.0], [-1.0, -2.0], [0.3, 0.2], [-0.2, -0.3]]) # stream with small edge examples.
y_a4 = np.array([1.0, 1.0, -1.0, -1.0, -1.0, 1.0]) # conflicting labels for edge cases.
X_val_a4 = np.array([[1.3, 1.1], [-1.3, -1.1], [0.5, 0.6], [-0.6, -0.5]]) # validation stream.
y_val_a4 = np.array([1.0, -1.0, 1.0, -1.0]) # validation labels.
print("stream size:", len(y_a4)) # inspect stream size.

In [ ]:
w_a4 = np.zeros(2) # current PA-I weights.
w_sum_a4 = np.zeros(2) # accumulator for averaged weights.
count_a4 = 0 # count stored iterates.
C_a4 = 0.5 # use a moderate cap.
for epoch_a4 in range(20): # train for multiple passes.
    for x_i_a4, y_i_a4 in zip(X_a4, y_a4): # scan examples.
        loss_i_a4 = max(0.0, 1.0 - y_i_a4 * float(w_a4 @ x_i_a4)) # compute hinge violation.
        tau_i_a4 = min(C_a4, loss_i_a4 / float(x_i_a4 @ x_i_a4)) if loss_i_a4 > 0 else 0.0 # capped PA update.
        w_a4 = w_a4 + tau_i_a4 * y_i_a4 * x_i_a4 # update current weights.
        w_sum_a4 = w_sum_a4 + w_a4 # accumulate after each online step.
        count_a4 += 1 # increment iterate count.
w_avg_a4 = w_sum_a4 / count_a4 # compute averaged online weights.
print("final w:", np.round(w_a4, 3), "avg w:", np.round(w_avg_a4, 3)) # inspect both models.

In [ ]:
val_final_a4 = float(np.mean(np.maximum(0.0, 1.0 - y_val_a4 * (X_val_a4 @ w_a4)))) # validation loss for last iterate.
val_avg_a4 = float(np.mean(np.maximum(0.0, 1.0 - y_val_a4 * (X_val_a4 @ w_avg_a4)))) # validation loss for averaged iterate.
print("validation final:", round(val_final_a4, 3), "validation averaged:", round(val_avg_a4, 3)) # compare losses.
assert val_avg_a4 >= 0.0 and val_final_a4 >= 0.0 # verify nonnegative hinge losses.

In [ ]:
plt.figure(figsize=(4, 3)) # create validation comparison chart.
plt.bar(["final iterate", "averaged iterate"], [val_final_a4, val_avg_a4], color=["gray", "teal"]) # compare validation losses.
plt.title("Advanced 4: averaged PA") # title chart.
plt.ylabel("validation hinge loss") # label scale.
plt.xticks(rotation=12) # fit labels.
plt.show() # display chart.

▶ What you'll see: averaging gives a second, often steadier model from the same online updates.

In [ ]:
plt.figure(figsize=(4, 3)) # create coordinate comparison.
plt.plot([0, w_a4[0]], [0, w_a4[1]], marker="o", label="final") # draw final vector.
plt.plot([0, w_avg_a4[0]], [0, w_avg_a4[1]], marker="s", label="averaged") # draw average vector.
plt.title("Advanced 4: final vs averaged weights") # title plot.
plt.xlabel("w0"); plt.ylabel("w1") # label axes.
plt.legend(); plt.show() # display chart.

▶ What you'll see: the averaged vector is a smoothed summary of the path taken by online learning.

👀 Takeaway: online algorithms can use iterate averaging to reduce variance without changing the basic PA update.

### Advanced 5 — Diagnose feature-scale sensitivity

**Goal.** Compare updates before and after feature scaling, because PA's denominator uses `||x||²` and therefore depends on coordinate scale. We build it in 5 steps.

In [ ]:
x_raw_a5 = np.array([100.0, 1.0]) # define an unscaled feature vector with one huge coordinate.
y_a5 = 1.0 # positive label.
w_a5 = np.array([0.0, 0.0]) # start at zero for a clean update.
loss_a5 = max(0.0, 1.0 - y_a5 * float(w_a5 @ x_raw_a5)) # compute hinge loss.
print("raw ||x||^2:", round(float(x_raw_a5 @ x_raw_a5), 3), "loss:", loss_a5) # inspect raw scale.

In [ ]:
tau_raw_a5 = loss_a5 / float(x_raw_a5 @ x_raw_a5) # compute PA step on unscaled features.
w_raw_a5 = w_a5 + tau_raw_a5 * y_a5 * x_raw_a5 # update raw features.
print("raw tau:", round(tau_raw_a5, 6), "raw update:", np.round(w_raw_a5, 6)) # inspect the tiny scalar and uneven coordinates.
assert round(float(y_a5 * (w_raw_a5 @ x_raw_a5)), 3) == 1.000 # verify exact margin repair.

In [ ]:
scale_a5 = np.array([100.0, 1.0]) # choose a simple known scaling divisor.
x_scaled_a5 = x_raw_a5 / scale_a5 # scale features to comparable magnitudes.
tau_scaled_a5 = loss_a5 / float(x_scaled_a5 @ x_scaled_a5) # compute PA step after scaling.
w_scaled_space_a5 = w_a5 + tau_scaled_a5 * y_a5 * x_scaled_a5 # update in scaled space.
print("scaled x:", x_scaled_a5, "scaled tau:", round(tau_scaled_a5, 3), "scaled-space w:", np.round(w_scaled_space_a5, 3)) # inspect scaled update.
assert round(tau_scaled_a5, 3) == 0.500 # verify 1 / (1^2+1^2).

In [ ]:
moves_a5 = np.array([np.linalg.norm(w_raw_a5), np.linalg.norm(w_scaled_space_a5)]) # compare update magnitudes in their own spaces.
print("update norms:", np.round(moves_a5, 3)) # inspect scale impact.
plt.figure(figsize=(4, 3)) # create scale comparison.
plt.bar(["raw features", "scaled features"], moves_a5, color=["crimson", "teal"]) # compare movement sizes.
plt.title("Advanced 5: scaling changes PA updates") # title chart.
plt.ylabel("||Δw||") # label scale.
plt.show() # display chart.

▶ What you'll see: feature scaling changes τ and the shape of the update, even with the same margin loss.

In [ ]:
plt.figure(figsize=(4, 3)) # create per-coordinate comparison.
plt.bar(["raw w0", "raw w1", "scaled w0", "scaled w1"], [w_raw_a5[0], w_raw_a5[1], w_scaled_space_a5[0], w_scaled_space_a5[1]], color=["crimson", "crimson", "teal", "teal"]) # compare coordinates.
plt.title("Advanced 5: coordinate effects") # title chart.
plt.xticks(rotation=20) # fit labels.
plt.show() # display chart.

▶ What you'll see: the huge raw coordinate dominates the norm, while scaled features distribute the update more evenly.

👀 Takeaway: because PA divides by `||x||²`, feature scaling is part of the algorithm's geometry, not a cosmetic preprocessing step.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Passive-aggressive learning changes weights only enough to fix the current margin violation.

Passive-aggressive algorithms are online margin methods. They stay passive when the current example already has margin, and become aggressive only enough to correct a violation. Save a copy to Drive to edit.

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))

    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))

    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))

    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))

    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))

    return rungs


def lesson_score(losses, cost, alternative):
    raw = float(np.sum(losses) / len(losses))
    score = raw + cost
    gap = alternative - score
    relative_gap = gap / alternative
    return raw, score, gap, relative_gap


def preview_ladder(rungs, is_regression=False):
    rows = []
    for index, item in enumerate(rungs, start=1):
        name, X, y = item
        if is_regression:
            info = f"target range {np.min(y):.2f}..{np.max(y):.2f}"
        else:
            values, counts = np.unique(y, return_counts=True)
            pairs = [f"{int(v)}:{int(c)}" for v, c in zip(values, counts)]
            info = ", ".join(pairs)
        row = {"rung": f"D{index}", "name": name, "shape": X.shape, "info": info}
        rows.append(row)
        print(row)
    name, X, y = rungs[0]
    print("sample X:")
    print(np.round(X[:5], 3))
    print("sample y:")
    print(np.round(y[:5], 3))
    return rows


def two_dimensional_view(X):
    if X.shape[1] == 1:
        return np.c_[X[:, 0], np.zeros(X.shape[0])]
    if X.shape[1] == 2:
        return X
    view = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(X))
    return view


def stream_batches(X, y, batch_size):
    rng = np.random.default_rng(11)
    order = rng.permutation(len(y))
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        yield X[idx], y[idx]


def online_fit_predict(X, y, kind="sgd", epochs=8, batch_size=16):
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=3,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    classes = np.unique(y)
    if kind == "pa":
        model = PassiveAggressiveClassifier(C=0.6, random_state=3, max_iter=1, tol=None)
    else:
        model = SGDClassifier(loss="log_loss", alpha=0.0005, random_state=3, learning_rate="optimal")
    first = True
    history = []
    batch_size = max(2, min(batch_size, len(y_train)))
    for epoch in range(epochs):
        for xb, yb in stream_batches(x_train, y_train, batch_size):
            if first:
                model.partial_fit(xb, yb, classes=classes)
                first = False
            else:
                model.partial_fit(xb, yb)
        preds = model.predict(x_test)
        history.append(float(accuracy_score(y_test, preds)))
    preds = model.predict(x_test)
    return model, scaler, x_train, x_test, y_train, y_test, preds, history


def logistic_accuracy(X, y, weighted=False):
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=4,
        stratify=stratify,
    )
    class_weight = None
    if weighted:
        class_weight = "balanced"
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=4),
    )
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    acc = float(accuracy_score(y_test, preds))
    return model, x_train, x_test, y_train, y_test, preds, acc


def expected_binary_cost(y_true, y_pred, false_negative_cost=5.0, false_positive_cost=1.0):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return float((false_negative_cost * fn + false_positive_cost * fp) / len(y_true))


def make_multi_targets(y):
    y = np.asarray(y, dtype=float)
    scale = np.std(y)
    if scale == 0:
        scale = 1.0
    centered = (y - np.mean(y)) / scale
    cuts = np.quantile(centered, [0.33, 0.66])
    ordinal = np.digitize(centered, cuts).astype(float)
    return np.c_[centered, ordinal]


def multioutput_fit_predict(X, y, alpha=1.0):
    targets = make_multi_targets(y)
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        targets,
        test_size=0.4,
        random_state=5,
    )
    model = make_pipeline(
        StandardScaler(),
        MultiOutputRegressor(Ridge(alpha=alpha)),
    )
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    mse = float(mean_squared_error(y_test, preds))
    r2 = float(r2_score(y_test, preds, multioutput="variance_weighted"))
    return model, x_train, x_test, y_train, y_test, preds, mse, r2


def make_survival_from_classification(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    rng = np.random.default_rng(23 + X.shape[0] + X.shape[1])
    weights = np.linspace(0.4, 1.2, X.shape[1])
    linear = StandardScaler().fit_transform(X).dot(weights) / math.sqrt(X.shape[1])
    class_effect = (y == np.max(y)).astype(float) * 0.8
    risk = linear + class_effect
    event_time = np.exp(-0.45 * risk) + rng.gamma(shape=2.0, scale=0.25, size=len(y))
    censor_time = rng.gamma(shape=2.3, scale=0.5, size=len(y)) + 0.35
    observed_time = np.minimum(event_time, censor_time)
    event = (event_time <= censor_time).astype(int)
    if np.sum(event) < 3:
        event[:3] = 1
    return observed_time, event


def cox_fit(X, time, event, lr=0.03, steps=220, l2=0.02):
    X = np.asarray(X, dtype=float)
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    beta = np.zeros(X.shape[1])
    order = np.argsort(-time)
    X_desc = X[order]
    event_desc = event[order]
    for step in range(steps):
        scores = np.clip(X_desc.dot(beta), -30, 30)
        exp_scores = np.exp(scores)
        risk_sum = np.cumsum(exp_scores)
        weighted_sum = np.cumsum(exp_scores[:, None] * X_desc, axis=0)
        grad = np.zeros_like(beta)
        event_positions = np.where(event_desc == 1)[0]
        for pos in event_positions:
            grad += X_desc[pos] - weighted_sum[pos] / risk_sum[pos]
        grad = grad / max(1, len(event_positions))
        grad = grad - l2 * beta
        beta = beta + lr * grad
    return beta


def concordance_index(time, event, risk):
    total = 0
    good = 0.0
    for i in range(len(time)):
        for j in range(len(time)):
            if time[i] < time[j] and event[i] == 1:
                total += 1
                if risk[i] > risk[j]:
                    good += 1.0
                elif risk[i] == risk[j]:
                    good += 0.5
    if total == 0:
        return 0.5
    return float(good / total)


def survival_fit_score(X, y):
    time, event = make_survival_from_classification(X, y)
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, t_train, t_test, e_train, e_test = train_test_split(
        X,
        time,
        event,
        test_size=0.4,
        random_state=6,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    beta = cox_fit(x_train, t_train, e_train)
    train_risk = x_train.dot(beta)
    test_risk = x_test.dot(beta)
    cindex = concordance_index(t_test, e_test, test_risk)
    return beta, scaler, x_train, x_test, t_train, t_test, e_train, e_test, train_risk, test_risk, cindex


def cross_validation_gap(X, y, k=5):
    counts = np.bincount(y.astype(int))
    min_count = int(np.min(counts))
    k = max(2, min(k, min_count))
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=8),
    )
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=8)
    result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    train_loss = 1.0 - result["train_score"]
    val_loss = 1.0 - result["test_score"]
    gap = float(np.mean(val_loss - train_loss))
    return train_loss, val_loss, gap, cv

## The concept, built once (D1)

The lesson formula is

$$w_{t+1}=w_t+\tau_t y_t x_t,\qquad \tau_t=\frac{\max(0,1-y_t w_t^\top x_t)}{\|x_t\|^2}$$

Plug in the lesson losses 0.235, 0.070, and 0.505. The average is $R_S=0.810/3=0.270$, the cost is $0.070$, the score is $0.340$, and the alternative gap is $0.384-0.340=0.044$.

In [ ]:
def passive_aggressive_algorithms_method():
    losses = np.array([0.235, 0.07, 0.505], dtype=float)
    cost = 0.070
    alternative = 0.384
    w = np.array([0.2, -0.1])
    x = np.array([2.0, 1.0])
    y = 1.0
    margin = y * np.dot(w, x)
    tau = max(0.0, 1.0 - margin) / np.dot(x, x)
    updated = w + tau * y * x
    raw, score, gap, relative_gap = lesson_score(losses, cost, alternative)
    assert np.isclose(tau, 0.14)
    assert np.isclose(raw, 0.270000000000)
    assert np.isclose(score, 0.340000000000)
    assert np.isclose(gap, 0.044000000000)
    return {"margin": margin, "tau": tau, "updated_weight": updated, "raw": raw, "score": score, "gap": gap}

lesson_check = passive_aggressive_algorithms_method()
print(lesson_check)

The method returns the arithmetic pieces and asserts the exact lesson numbers before any larger data appears.

In [ ]:
assert lesson_check['score'] > lesson_check['raw']
assert lesson_check['gap'] > 0
print('lesson arithmetic locked')

## The dataset ladder

Use the shared classification ladder so the same learner runs from a hand toy to real Breast Cancer features.

In [ ]:
rungs = clf_ladder()
ladder_preview = preview_ladder(rungs, is_regression=False)

## Run the same method across D1–D5

Only the data rung changes. The metric is the plan metric for this topic.

In [ ]:
results = []
artifacts = []
for rung_index, (name, X, y) in enumerate(rungs, start=1):
    model, scaler, x_train, x_test, y_train, y_test, preds, history = online_fit_predict(X, y, kind="pa")
    acc = float(accuracy_score(y_test, preds))
    results.append({"rung": rung_index, "name": name, "accuracy": acc, "last_stream_accuracy": history[-1]})
    artifacts.append((name, X, y, preds, history, model, scaler))
for row in results:
    print(f"D{row['rung']} {row['accuracy']:.3f} accuracy — {row['name']}")

## Results visualization

The first figure shows the model artifact on each rung. The second summarizes `accuracy` from D1 through D5.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for index, artifact in enumerate(artifacts):
    name, X, y, preds, history, model, scaler = artifact
    view = two_dimensional_view(X)
    axes[0, index].scatter(view[:, 0], view[:, 1], c=y, cmap="viridis", s=18, edgecolor="k", linewidth=0.2)
    axes[0, index].set_title(f"D{index + 1}: {name.split('(')[0]}")
    axes[0, index].set_xticks([])
    axes[0, index].set_yticks([])
    axes[1, index].plot(history, marker="o")
    axes[1, index].set_ylim(0, 1.05)
    axes[1, index].set_title("stream accuracy")
    axes[1, index].set_xlabel("epoch")
axes[1, 0].set_ylabel("accuracy")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3.5))
plt.plot([row["rung"] for row in results], [row["accuracy"] for row in results], marker="o")
plt.xticks([1, 2, 3, 4, 5], ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0, 1.05)
plt.ylabel("accuracy")
plt.title("accuracy vs. ladder complexity")
plt.grid(True, alpha=0.3)
plt.show()

## Pitfall on the hardest rung

The lesson warning is to optimize the raw term and forget the cost. On D5, the raw metric alone can pick a different setting than the cost-aware score.

In [ ]:
name, X, y = rungs[-1]
small_c = online_fit_predict(X, y, kind="pa", epochs=8, batch_size=16)
large_c = online_fit_predict(X, y, kind="pa", epochs=8, batch_size=4)
small_acc = small_c[7][-1]
large_acc = large_c[7][-1]
raw_only_winner = "large-updates" if large_acc >= small_acc else "moderate-updates"
large_score = (1.0 - large_acc) + 0.070
small_score = (1.0 - small_acc) + 0.070 * 0.5
cost_aware_winner = "large-updates" if large_score <= small_score else "moderate-updates"
print("D5 raw accuracies", large_acc, small_acc, "raw winner", raw_only_winner)
print("D5 cost-aware scores", large_score, small_score, "cost-aware winner", cost_aware_winner)
print("lesson raw", 0.270, "cost", 0.070, "score", 0.340, "gap", 0.384 - 0.340)

## Evaluate it + Practice

- Compare the displayed metric with a no-skill baseline such as majority class, mean target, or random fold assignment.
- Sanity-check D1 by hand before trusting the D5 curve.
- Ablate the key idea: remove partial updates, remove PA margins, collapse outputs, ignore censoring, remove costs, or reuse the test set.
- Failure signals include unstable D5 metrics, a widening validation gap, or a cost-aware score that disagrees with the raw metric.

Practice 1: change the seed or batch/fold size and rerun the D1-to-D5 table.

Practice 2: turn off the topic-specific idea and measure the metric drop on D5.

Practice 3: add one extra diagnostic plot for the hardest rung.